In [1]:
import pandas as pd
import numpy as np


In [2]:
df_raw = pd.read_excel("realistic_modified_student_performance.xlsx")
df = df_raw.copy()
df = df.sort_values(['Student_ID', 'Semester'])

In [3]:
df.head()

,Student_ID,Student_Name,Batch,Semester,Year,Mid1,Mid2,Internals,EndTerm,Attendance,Total_Marks,SGPA,Performance_Type,Placement
0,S0001,Neha Pal,2018,1,2018,9.82,5.10,11.46,73.87,78.8,63.32,6.09,Low_Performer,Not Placed
1,S0001,Neha Pal,2018,2,2018,7.48,7.55,9.98,54.82,91.4,52.42,6.31,Low_Performer,Not Placed
2,S0001,Neha Pal,2018,3,2019,9.16,10.36,8.06,42.76,87.8,48.97,6.59,Low_Performer,Not Placed
3,S0001,Neha Pal,2018,4,2019,6.56,10.50,15.56,48.40,87.2,56.82,6.05,Low_Performer,Not Placed
4,S0001,Neha Pal,2018,5,2020,11.19,4.43,9.45,48.62,83.7,49.38,5.39,Low_Performer,Not Placed


In [4]:
# Recreate rolling dip flag
df_roll = df.copy()
df_roll['Rolling_Avg'] = df_roll.groupby('Student_ID')['SGPA'].transform(
    lambda x: x.expanding().mean()
)
df_roll['Rolling_STD'] = df_roll.groupby('Student_ID')['SGPA'].transform(
    lambda x: x.expanding().std()
).fillna(0)

In [5]:
df_roll['Is_Dip_Roll'] = df_roll['SGPA'] < (df_roll['Rolling_Avg'] - df_roll['Rolling_STD'])

In [6]:
print(df_roll.shape)  

(2752, 17)


In [7]:
df_roll.head()

,Student_ID,Student_Name,Batch,Semester,Year,Mid1,Mid2,Internals,EndTerm,Attendance,Total_Marks,SGPA,Performance_Type,Placement,Rolling_Avg,Rolling_STD,Is_Dip_Roll
0,S0001,Neha Pal,2018,1,2018,9.82,5.10,11.46,73.87,78.8,63.32,6.09,Low_Performer,Not Placed,6.090,0.000000,False
1,S0001,Neha Pal,2018,2,2018,7.48,7.55,9.98,54.82,91.4,52.42,6.31,Low_Performer,Not Placed,6.200,0.155563,False
2,S0001,Neha Pal,2018,3,2019,9.16,10.36,8.06,42.76,87.8,48.97,6.59,Low_Performer,Not Placed,6.330,0.250599,False
3,S0001,Neha Pal,2018,4,2019,6.56,10.50,15.56,48.40,87.2,56.82,6.05,Low_Performer,Not Placed,6.260,0.247925,False
4,S0001,Neha Pal,2018,5,2020,11.19,4.43,9.45,48.62,83.7,49.38,5.39,Low_Performer,Not Placed,6.086,0.444387,True


Collasping all Sems rows in single Row

In [8]:
def build_student_features(student_df):
    student_df = student_df.sort_values("Semester")
    sgpa = student_df["SGPA"].values
    sems = student_df["Semester"].values

    # Slop for Trend
    if len(sgpa)>1:
        polyfit = np.polyfit(sems, sgpa, 1)
        slope = polyfit[0]
    else:
        slope = 0.0


    return pd.Series({
        'current_sgpa' : sgpa[-1],
        'sgpa_mean' : sgpa.mean(),
        "sgpa_trend" : round(slope, 4),
        'consistency_score' : sgpa.std(),
        'dip_count': int(student_df['Is_Dip_Roll'].sum()),
        'last_sem_dip': int(student_df['Is_Dip_Roll'].iloc[-1]),
        'attendance_mean': student_df['Attendance'].mean(),
        'internals_avg':     student_df['Internals'].mean(),
        
        'placement_ready' : student_df["Placement"].iloc[0]
    })
        
# Apply → 224 rows (one per student)
df_features = df_roll.groupby('Student_ID').apply(build_student_features).reset_index()


C:\Users\ani77\AppData\Local\Temp\ipykernel_7528\401081819.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_features = df_roll.groupby('Student_ID').apply(build_student_features).reset_index()


In [9]:
print(df_features.shape)  # (224, 11)


(344, 10)


In [10]:
df_features.head()

,Student_ID,current_sgpa,sgpa_mean,sgpa_trend,consistency_score,dip_count,last_sem_dip,attendance_mean,internals_avg,placement_ready
0,S0001,5.03,5.89500,-0.1560,0.480833,3,1,83.4750,9.77250,Not Placed
1,S0002,7.15,7.59500,-0.0452,0.492874,0,0,72.7750,12.14000,Not Placed
2,S0003,7.90,7.07000,0.1838,0.482986,0,0,77.2500,11.51250,Not Placed
3,S0004,5.11,5.33875,-0.1763,0.571849,3,0,81.0125,9.85625,Not Placed
4,S0005,5.39,6.02625,-0.2235,0.553284,4,1,82.8375,10.76875,Placed


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [12]:
FEATURES = [
    "current_sgpa", "sgpa_mean", "sgpa_trend", "consistency_score", "dip_count", 'last_sem_dip',
    'attendance_mean', 'internals_avg'
]

X = df_features[FEATURES]
y = df_features["placement_ready"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [13]:
print(classification_report(y_test, model.predict(X_test),
      target_names=['Not Ready', 'Placement Ready']))

                 precision    recall  f1-score   support

      Not Ready       0.74      0.82      0.78        34
Placement Ready       0.81      0.71      0.76        35

       accuracy                           0.77        69
      macro avg       0.77      0.77      0.77        69
   weighted avg       0.77      0.77      0.77        69



In [14]:
from sklearn.metrics import accuracy_score
y_pred = model.predict(X_test)
acc_RandomFor = accuracy_score(y_test, y_pred)
print(acc_RandomFor)

0.7681159420289855


In [15]:
importance = pd.Series(
    model.feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

print(importance)

internals_avg        0.223005
current_sgpa         0.179413
sgpa_mean            0.168227
sgpa_trend           0.147328
consistency_score    0.129051
attendance_mean      0.097888
dip_count            0.043574
last_sem_dip         0.011513
dtype: float64


### Predicting a students v

In [16]:
def predict_student_readiness(student_df, model):

    # Build features automatically
    features = build_student_features(student_df)

    # Convert into dataframe
    features_df = pd.DataFrame([features])

    # Keep only model features
    X_input = features_df[FEATURES]

    # Predict probability
    prob = model.predict_proba(X_input)[0][1]

    # Predict class
    pred = model.predict(X_input)[0]

    return {
        "prediction": pred,
        "placement_probability": round(prob, 4)
    }

In [17]:
student_data = pd.DataFrame({

    "Semester": [1, 2, 3],

    "SGPA": [7.45, 8.12, 7.80],

    "Attendance": [82, 85, 80],

    "Internals": [21, 24, 23],

})

In [18]:
result = predict_student_readiness(student_data, model)

print(result)

KeyError: 'Is_Dip_Roll'

## OTHER MODELS

### Logistic

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)

In [ ]:
y_pred = model.predict(X_test_scaled)
acc_logis = accuracy_score(y_test, y_pred)
print(acc_logis)

### XGBOOST

In [ ]:
from xgboost import XGBClassifier
model = XGBClassifier(n_estimators=100,max_depth=4,learning_rate=0.1,random_state=42,eval_metric='logloss'
)



y_xg = df_features["placement_ready"].map({
    "Not Placed": 0,
    "Placed": 1
})

X = df_features[FEATURES]
y = y_xg

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_xg,
    test_size=0.2,
    random_state=42,
    stratify=y_xg
)


model.fit(X_train, y_train)
y_pred = model.predict(X_test)


In [ ]:
acc_XG = accuracy_score(y_test, y_pred)

acc_XG

### SVM

In [ ]:
from sklearn.svm import SVC


X = df_features[FEATURES]
# Binary target
y_svm = df_features["placement_ready"].map({
    "Not Placed": 0,
    "Placed": 1
})

X_train, X_test, y_train, y_test = train_test_split(
    X,y_svm,test_size=0.2,random_state=42,stratify=y_svm
)



In [ ]:
model = SVC(kernel='rbf',C=1.0,gamma='scale',random_state=42)

model.fit(X_train_scaled, y_train)

In [ ]:
y_pred = model.predict(X_test_scaled)


acc_svm = accuracy_score(y_test, y_pred)

acc_svm

In [ ]:
print(acc_svm, acc_logis, acc_RandomFor, acc_XG)

In [ ]:
print(df_features["placement_ready"].value_counts())